<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week9_Day3_Exercise_XP_de_agentic_ai_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1) Build the KB retriever

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

/tmp/ipykernel_807/1722337650.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


KB ready with 5 docs


In [ ]:
import pandas as pd

# Expanding the KB to 8 documents as requested in the Exercise XP
kb_docs.extend([
    Document(
        page_content="""
Citation accuracy is critical. Every claim derived from external knowledge should have a corresponding tag
like [kb:source_name] or [wiki:Title] to allow for human verification.
""".strip(),
        metadata={"source": "kb:citation_rules"},
    ),
    Document(
        page_content="""
Fallback mechanisms ensure that when primary data sources fail or return no results, the system can
still provide value by searching broader indices like Wikipedia or providing a helpful error message.
""".strip(),
        metadata={"source": "kb:fallback_logic"},
    ),
    Document(
        page_content="""
Query normalization involves removing stop words and expanding synonyms to ensure the retriever
can match the user's intent even if the exact phrasing differs from the index.
""".strip(),
        metadata={"source": "kb:query_prep"},
    ),
])

# Re-initializing the vector store with the expanded KB
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("Expanded KB ready with", len(kb_docs), "docs")

Expanded KB ready with 8 docs


## 2) Open source external tool: Wikipedia search

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    try:
        results = wiki.results(query)
        snippets = [{'title': r['title'], 'summary': r['summary']} for r in results[:k]]
        return snippets, None
    except Exception as e:
        return [], str(e)

print(wiki_search('Python programming')[0][:1])

[{'title': 'Python (programming language)', 'summary': 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revi'}]


## 3) Simple planner (rule-based)

In [ ]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest']

def plan(question: str):
    q_lower = question.lower()
    if any(kw in q_lower for kw in kb_keywords):
        return {'action': 'kb'}
    return {'action': 'wiki'}

print(plan('How to ground answers?'))
print(plan('Who created Python?'))

{'action': 'kb'}
{'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets.\n"
    "If there is little evidence, say so and suggest a follow-up query.\n"
    "Cite sources like [kb:doc1] or [wiki:Title].\n"
    "Question: {question}\n"
    "Context:{context}\n"
    "Wiki:{wiki}\n"
    "Answer:"
)

def get_tiny_generator(model_id: str = 'sshleifer/tiny-gpt2'):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline('text-generation', model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    gen = get_tiny_generator()
    out = gen(prompt_text, max_new_tokens=max_new_tokens, pad_token_id=50256, truncation=True)
    full = out[0]["generated_text"]
    completion = full[len(prompt_text):].strip()
    return completion

def answer_question(question: str, use_tiny_model: bool = False):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action']=='kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action']=='wiki':
        wiki_snips, wiki_err = wiki_search(question)
    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)
    else:
        stub = FakeListChatModel(responses=['Stub summary based on provided context and wiki snippets.'])
        final_answer = stub.invoke(messages).content
    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

## 5) Quick check on sample questions

In [ ]:
tests = [
    "What are agentic systems?",
    "Who is the CEO of Apple?",
    "How to handle honesty in RAG?"
]

for q in tests:
    res = answer_question(q, use_tiny_model=False)
    print('---')
    print('Q:', q)
    print('Plan:', res['plan'])
    print('KB sources:', res['kb_sources'])
    print('Wiki sources:', res['wiki_sources'])
    print('Answer:', res['answer'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Q: How do agentic systems decide when to use tools?
Answer: How do you use agentic systems when deciding when to use specific tools, and what principles should be followed?


Device set to use cpu


Q: Who created Python and when?
Answer: The question is asking about the creation of Python, specifically who created Python. The answer is: Guido van Rossmus began working on the language in the late '80s and it was released in 1008.
Guide: Python is a programming language designed for structured programming with strong type checking and garbage collection. It is primarily used in Python 2.7. Python 2 is still in active development and will likely continue to be supported for several years to come. Python 0.9 was released in July 1991, and Python


Device set to use cpu


Q: What should I do if evidence is missing in RAG?
Answer: The Srebrencians have been accused of committing the massacre by the International Criminal Tribunal for the former Yugoslavia (ICTY) and the Bosnijsko-Hercegovinska Republica (BHKR) for the past 14 years. The massacres took place in 1945 and 1971 and resulted in the deaths of approximately 8,300 Bosnian Muslim men and boy, and their families. In 2003, the ICTY sentenced a


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from langchain_community.utilities import WikipediaAPIWrapper

# 1. Wikipedia Tool Setup
wiki_wrapper = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    try:
        results = wiki_wrapper.results(query)
        snippets = [{'title': r['title'], 'summary': r['summary']} for r in results[:k]]
        return snippets, None
    except Exception as e:
        return [], str(e)

# 2. Planner Logic
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest']

def plan(question: str):
    q_lower = question.lower()
    if any(kw in q_lower for kw in kb_keywords):
        return {'action': 'kb'}
    return {'action': 'wiki'}

# 3. Answer Function Template
prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets.\n"
    "If there is little evidence, say so and suggest a follow-up query.\n"
    "Cite sources like [kb:doc1] or [wiki:Title].\n"
    "Question: {question}\n"
    "Context:{context}\n"
    "Wiki:{wiki}\n"
    "Answer:"
)

def answer_question(question: str, use_tiny_model: bool = False):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action'] == 'kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action'] == 'wiki':
        wiki_snips, wiki_err = wiki_search(question)

    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'

    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    stub = FakeListChatModel(responses=['Stub summary based on provided context and wiki snippets.'])
    final_answer = stub.invoke(messages).content

    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'answer': final_answer,
    }

# 4. Final Exercise XP Check
final_tests = [
    "What are the core steps of an agentic loop?",
    "Who is the current Prime Minister of Japan?",
    "What is the best way to handle ambiguous RAG evidence?"
]

for q in final_tests:
    res = answer_question(q, use_tiny_model=False)
    print('\n' + '='*20)
    print(f'QUESTION: {q}')
    print(f'PLAN: {res["plan"]}')
    print(f'SOURCES: {res["kb_sources"] + res["wiki_sources"]}')
    print(f'ANSWER: {res["answer"]}')


QUESTION: What are the core steps of an agentic loop?
PLAN: {'action': 'kb'}
SOURCES: ['kb:citation_rules', 'kb:wikipedia_tip', 'kb:retrievers']
ANSWER: Stub summary based on provided context and wiki snippets.

QUESTION: Who is the current Prime Minister of Japan?
PLAN: {'action': 'wiki'}
SOURCES: []
ANSWER: Stub summary based on provided context and wiki snippets.

QUESTION: What is the best way to handle ambiguous RAG evidence?
PLAN: {'action': 'wiki'}
SOURCES: []
ANSWER: Stub summary based on provided context and wiki snippets.
